In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Filter and reduce columns early, compute TMP before merges
date1 = pd.Timestamp("1994-11-01")
date2 = pd.Timestamp("1995-02-01")

# lineitem: keep only returned items and needed columns, compute TMP
line_f = (
    lineitem.loc[lineitem.L_RETURNFLAG == "R", ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]]
    .assign(TMP=lambda df: df.L_EXTENDEDPRICE * (1.0 - df.L_DISCOUNT))
    [["L_ORDERKEY", "TMP"]]
)

# orders: filter by date and keep only join keys
ord_f = orders.loc[
    (orders.O_ORDERDATE >= date1) & (orders.O_ORDERDATE < date2),
    ["O_ORDERKEY", "O_CUSTKEY"]
]

# join lineitem -> orders
df = line_f.merge(
    ord_f,
    left_on="L_ORDERKEY",
    right_on="O_ORDERKEY",
    how="inner"
)[["O_CUSTKEY", "TMP"]]

# customer: select needed columns
cust_cols = [
    "C_CUSTKEY", "C_NAME", "C_ACCTBAL", "C_PHONE",
    "C_NATIONKEY", "C_ADDRESS", "C_COMMENT"
]

# join customer
df = df.merge(
    customer[cust_cols],
    left_on="O_CUSTKEY",
    right_on="C_CUSTKEY",
    how="inner"
).drop(columns=["O_CUSTKEY"])

# nation: select needed columns and join
nat_f = nation[["N_NATIONKEY", "N_NAME"]]
df = df.merge(
    nat_f,
    left_on="C_NATIONKEY",
    right_on="N_NATIONKEY",
    how="inner"
).drop(columns=["C_NATIONKEY", "N_NATIONKEY"])

# final aggregation and sort
total = (
    df
    .groupby(
        [
            "C_CUSTKEY", "C_NAME", "C_ACCTBAL", "C_PHONE",
            "N_NAME", "C_ADDRESS", "C_COMMENT"
        ],
        as_index=False,
        sort=False
    )["TMP"].sum()
    .sort_values("TMP", ascending=False)
)